In [1]:
"""
Sort non-high-confidence funds into separate files by category.
Each file includes the full data rows + a summary ID list for tracing.

Categories:
  1. json_parse_errors (115) — Model responded but JSON malformed. Recoverable with retry.
  2. api_errors (54)         — Request never completed (credit/rate limit). Must retry.
  3. no_objectives (130)     — Pass 1-2 found nothing. May need manual review of source text.
  4. medium_confidence (65)  — Extracted but verifier flagged uncertainty. Manual review.
  5. low_confidence (5)      — Extracted but verifier has serious doubts. Manual review.
  6. high_flagged (10)       — High confidence overall but ≥1 objective couldn't be verified.
"""

import pandas as pd
import os

# === CONFIG ===
INPUT_FILE = "REVIEW_non_high_confidence_379_funds.xlsx"  # UPDATE if needed
OUTPUT_DIR = "review_sorted"  # subfolder for sorted files

os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_excel(INPUT_FILE)
print(f"Loaded {len(df)} funds")

# === CATEGORISE ===
categories = {}

# 1. JSON parse failures (error confidence, summary starts with "JSON parse fail")
mask_json = (df['Overall_Confidence'] == 'error') & \
            (df['Verification_Summary'].str.startswith('JSON parse fail', na=False))
categories['1_json_parse_errors'] = df[mask_json].copy()

# 2. API errors (error confidence, not JSON parse — credit balance, rate limit, etc.)
mask_api = (df['Overall_Confidence'] == 'error') & ~mask_json
categories['2_api_errors'] = df[mask_api].copy()

# 3. No objectives found (none confidence)
mask_none = df['Overall_Confidence'] == 'none'
categories['3_no_objectives'] = df[mask_none].copy()

# 4. Medium confidence
mask_medium = df['Overall_Confidence'] == 'medium'
categories['4_medium_confidence'] = df[mask_medium].copy()

# 5. Low confidence
mask_low = df['Overall_Confidence'] == 'low'
categories['5_low_confidence'] = df[mask_low].copy()

# 6. High confidence but has flagged objectives
mask_high_flagged = (df['Overall_Confidence'] == 'high') & (df['Has_Flagged'] == True)
categories['6_high_but_flagged'] = df[mask_high_flagged].copy()

# === SAVE FILES ===
print(f"\n{'Category':<30} {'Count':>6}   File")
print("-" * 80)

all_id_rows = []

for name, cat_df in categories.items():
    if len(cat_df) == 0:
        continue

    # Save full data
    filename = f"{name}_{len(cat_df)}_funds.xlsx"
    filepath = os.path.join(OUTPUT_DIR, filename)
    cat_df.to_excel(filepath, index=False, engine='openpyxl')

    # Collect IDs for the summary
    for _, row in cat_df.iterrows():
        all_id_rows.append({
            'FundId': row['FundId'],
            'Fund_Name': row['Fund_Name'],
            'Category': name,
            'Overall_Confidence': row['Overall_Confidence'],
            'Has_Flagged': row['Has_Flagged'],
            'Number_of_Objectives': row['Number_of_Objectives']
        })

    print(f"  {name:<30} {len(cat_df):>4}   {filename}")

# === SAVE MASTER ID LIST ===
id_df = pd.DataFrame(all_id_rows)
id_filename = "ALL_review_fund_ids.xlsx"
id_path = os.path.join(OUTPUT_DIR, id_filename)
id_df.to_excel(id_path, index=False, engine='openpyxl')

# Also save just the retry IDs (json parse + api errors) as a flat list
retry_ids = id_df[id_df['Category'].isin(['1_json_parse_errors', '2_api_errors'])]['FundId'].tolist()
retry_df = pd.DataFrame({'FundId': retry_ids})
retry_path = os.path.join(OUTPUT_DIR, "RETRY_fund_ids.xlsx")
retry_df.to_excel(retry_path, index=False, engine='openpyxl')

print(f"\n{'='*80}")
print(f"SUMMARY")
print(f"{'='*80}")
print(f"  Total categorised: {len(id_df)}")
print(f"  Retryable (json + api errors): {len(retry_ids)}")
print(f"  Manual review needed: {len(id_df) - len(retry_ids)}")
print(f"\n  Master ID list:  {id_filename}")
print(f"  Retry ID list:   RETRY_fund_ids.xlsx")
print(f"\n  All files in:    {os.path.abspath(OUTPUT_DIR)}/")

# === SANITY CHECK ===
total_categorised = sum(len(v) for v in categories.values())
if total_categorised != len(df):
    uncategorised = len(df) - total_categorised
    print(f"\n  WARNING: {uncategorised} funds not categorised (check filters)")
else:
    print(f"\n  All {len(df)} funds categorised.")

Loaded 379 funds

Category                        Count   File
--------------------------------------------------------------------------------
  1_json_parse_errors             115   1_json_parse_errors_115_funds.xlsx
  2_api_errors                     54   2_api_errors_54_funds.xlsx
  3_no_objectives                 130   3_no_objectives_130_funds.xlsx
  4_medium_confidence              65   4_medium_confidence_65_funds.xlsx
  5_low_confidence                  5   5_low_confidence_5_funds.xlsx
  6_high_but_flagged               10   6_high_but_flagged_10_funds.xlsx

SUMMARY
  Total categorised: 379
  Retryable (json + api errors): 169
  Manual review needed: 210

  Master ID list:  ALL_review_fund_ids.xlsx
  Retry ID list:   RETRY_fund_ids.xlsx

  All files in:    /Users/dannyhogan/Desktop/Hogan_RA_Work/review_sorted/

  All 379 funds categorised.
